# Week 3: Train Uncertainty Type Classifier (Diverse Dataset)

## Goal

Train a robust probe to classify uncertainty type:
- **Code uncertainty** (label=1): Model uncertain about which code element to use
- **Language uncertainty** (label=0): Model uncertain about which word/phrase to use

## Dataset Size: 100+ Examples

Diverse across:
- Programming languages (Python, JS, Rust, Go, Java, C++, etc.)
- Domains (ML, web, systems, databases, DevOps)
- Difficulty levels (beginner to advanced)
- Code patterns (imports, functions, classes, loops, etc.)
- Language patterns (explanations, questions, creative writing)

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

## 1. Diverse Training Dataset (100+ Examples)

In [ ]:
# Cell 4: Training examples - CODE UNCERTAINTY (label = 1)

CODE_UNCERTAINTY_EXAMPLES = [
    # ========== PYTHON ==========
    {'prompt': 'import', 'desc': 'Python import - uncertain module'},
    {'prompt': 'from numpy import', 'desc': 'NumPy import - uncertain function'},
    {'prompt': 'import pandas as pd\ndf = pd.', 'desc': 'Pandas method - uncertain'},
    {'prompt': 'def calculate_', 'desc': 'Function name - uncertain'},
    {'prompt': 'class DataProcessor:', 'desc': 'Class body - uncertain method'},
    {'prompt': 'for i in range(10):\n    ', 'desc': 'Loop body - uncertain statement'},
    {'prompt': 'if user.is_', 'desc': 'Attribute access - uncertain'},
    {'prompt': 'result = requests.', 'desc': 'HTTP method - uncertain'},
    {'prompt': 'model = tf.keras.', 'desc': 'Keras class - uncertain'},
    {'prompt': 'with open(file) as f:\n    data = f.', 'desc': 'File method - uncertain'},
    
    # ========== JAVASCRIPT/TYPESCRIPT ==========
    {'prompt': 'const [state, setState] = use', 'desc': 'React hook - uncertain'},
    {'prompt': 'import { Component } from', 'desc': 'ES6 import - uncertain package'},
    {'prompt': 'async function fetchData() {\n    await', 'desc': 'Async operation - uncertain'},
    {'prompt': 'const data = await fetch(url).', 'desc': 'Fetch method - uncertain'},
    {'prompt': 'app.', 'desc': 'Express method - uncertain'},
    {'prompt': 'export default', 'desc': 'Export - uncertain identifier'},
    {'prompt': 'interface User {\n    name: string;\n    ', 'desc': 'Interface property - uncertain'},
    {'prompt': 'function handleClick', 'desc': 'Event handler - uncertain params'},
    
    # ========== RUST ==========
    {'prompt': 'use std::', 'desc': 'Rust use - uncertain module'},
    {'prompt': 'fn main() {', 'desc': 'Rust function - uncertain body'},
    {'prompt': 'let mut vec = Vec::', 'desc': 'Vec method - uncertain'},
    {'prompt': 'impl Trait for', 'desc': 'Trait impl - uncertain type'},
    {'prompt': 'match result {\n    Ok(val) => ', 'desc': 'Match arm - uncertain'},
    
    # ========== GO ==========
    {'prompt': 'package main\nimport "', 'desc': 'Go import - uncertain package'},
    {'prompt': 'func main() {', 'desc': 'Go function - uncertain body'},
    {'prompt': 'type User struct {\n    Name string\n    ', 'desc': 'Struct field - uncertain'},
    {'prompt': 'go func() {', 'desc': 'Goroutine - uncertain body'},
    
    # ========== JAVA ==========
    {'prompt': 'import java.util.', 'desc': 'Java import - uncertain class'},
    {'prompt': 'public class Main extends', 'desc': 'Class extends - uncertain parent'},
    {'prompt': 'private static final', 'desc': 'Field declaration - uncertain type'},
    {'prompt': '@Override\npublic void', 'desc': 'Method override - uncertain name'},
    
    # ========== C++ ==========
    {'prompt': '#include <', 'desc': 'C++ include - uncertain header'},
    {'prompt': 'std::', 'desc': 'STL - uncertain class'},
    {'prompt': 'template<typename T>\nclass', 'desc': 'Template class - uncertain name'},
    
    # ========== SQL ==========
    {'prompt': 'SELECT * FROM', 'desc': 'SQL table - uncertain'},
    {'prompt': 'SELECT name, email FROM users WHERE', 'desc': 'SQL condition - uncertain'},
    {'prompt': 'CREATE TABLE users (\n    id INT PRIMARY KEY,\n    ', 'desc': 'SQL column - uncertain'},
    {'prompt': 'INSERT INTO users (name, email) VALUES', 'desc': 'SQL values - uncertain'},
    {'prompt': 'UPDATE users SET', 'desc': 'SQL update - uncertain column'},
    
    # ========== BASH/SHELL ==========
    {'prompt': 'git ', 'desc': 'Git command - uncertain'},
    {'prompt': 'docker run -', 'desc': 'Docker flag - uncertain'},
    {'prompt': 'kubectl apply -f', 'desc': 'Kubectl file - uncertain'},
    {'prompt': 'ssh user@', 'desc': 'SSH host - uncertain'},
    {'prompt': 'curl -X POST', 'desc': 'Curl request - uncertain URL'},
    
    # ========== ML/DATA SCIENCE ==========
    {'prompt': 'import torch.nn.', 'desc': 'PyTorch module - uncertain'},
    {'prompt': 'from sklearn.', 'desc': 'Sklearn module - uncertain'},
    {'prompt': 'model = Sequential([\n    Dense(128, activation=', 'desc': 'Activation - uncertain'},
    {'prompt': 'X_train, X_test = train_test_split(X, y, test_size=', 'desc': 'Split ratio - uncertain'},
    {'prompt': 'plt.', 'desc': 'Matplotlib function - uncertain'},
    
    # ========== WEB FRAMEWORKS ==========
    {'prompt': 'app = FastAPI()\n@app.', 'desc': 'FastAPI decorator - uncertain'},
    {'prompt': '@app.route(\'/api/', 'desc': 'Flask route - uncertain path'},
    {'prompt': 'router.', 'desc': 'Express router method - uncertain'},
    
    # ========== DATABASES ==========
    {'prompt': 'connection = psycopg2.', 'desc': 'psycopg2 method - uncertain'},
    {'prompt': 'db.collection(', 'desc': 'MongoDB collection - uncertain'},
    {'prompt': 'redis_client.', 'desc': 'Redis method - uncertain'},
    
    # ========== CLOUD/INFRA ==========
    {'prompt': 'aws s3 cp', 'desc': 'AWS CLI - uncertain path'},
    {'prompt': 'terraform apply -var-file=', 'desc': 'Terraform var file - uncertain'},
    {'prompt': 'ansible-playbook -i', 'desc': 'Ansible inventory - uncertain'},
]

print(f"Code uncertainty examples: {len(CODE_UNCERTAINTY_EXAMPLES)}")

In [ ]:
# Cell 5: Training examples - LANGUAGE UNCERTAINTY (label = 0)

LANGUAGE_UNCERTAINTY_EXAMPLES = [
    # ========== TECHNICAL EXPLANATIONS ==========
    {'prompt': 'This function', 'desc': 'Verb choice - uncertain'},
    {'prompt': 'The algorithm is', 'desc': 'Adjective choice - uncertain'},
    {'prompt': 'The main advantage of microservices is', 'desc': 'Explanation - uncertain'},
    {'prompt': 'Machine learning models can', 'desc': 'Capability description - uncertain'},
    {'prompt': 'Docker containers are useful because they', 'desc': 'Reason - uncertain'},
    {'prompt': 'The key benefit of async programming is', 'desc': 'Benefit - uncertain'},
    {'prompt': 'TypeScript provides better', 'desc': 'Improvement - uncertain'},
    {'prompt': 'Code quality can be', 'desc': 'Quality measure - uncertain'},
    {'prompt': 'Database normalization helps', 'desc': 'Help description - uncertain'},
    {'prompt': 'REST APIs are designed to', 'desc': 'Purpose - uncertain'},
    
    # ========== COMPARISONS ==========
    {'prompt': 'The difference between REST and GraphQL is', 'desc': 'Comparison - uncertain'},
    {'prompt': 'Unlike synchronous code, asynchronous code', 'desc': 'Contrast - uncertain'},
    {'prompt': 'Compared to NoSQL, SQL databases', 'desc': 'Comparison - uncertain'},
    {'prompt': 'While Python is', 'desc': 'Characteristic - uncertain'},
    {'prompt': 'Rust differs from C++ in that it', 'desc': 'Difference - uncertain'},
    
    # ========== INSTRUCTIONS/QUESTIONS ==========
    {'prompt': 'Explain what this code', 'desc': 'Explanation verb - uncertain'},
    {'prompt': 'Explain how binary search', 'desc': 'Explanation - uncertain'},
    {'prompt': 'Describe the process of', 'desc': 'Description - uncertain'},
    {'prompt': 'What are the benefits of', 'desc': 'Benefits question - uncertain'},
    {'prompt': 'How does garbage collection', 'desc': 'Process question - uncertain'},
    {'prompt': 'Why is immutability', 'desc': 'Reason question - uncertain'},
    {'prompt': 'When should you use', 'desc': 'Usage question - uncertain'},
    
    # ========== DOCUMENTATION/COMMENTS ==========
    {'prompt': 'This method calculates', 'desc': 'Documentation - uncertain verb'},
    {'prompt': 'Returns the', 'desc': 'Return description - uncertain'},
    {'prompt': 'Parameters:', 'desc': 'Parameter description - uncertain'},
    {'prompt': 'Note that this implementation', 'desc': 'Note - uncertain'},
    {'prompt': 'The following code demonstrates', 'desc': 'Demonstration - uncertain'},
    
    # ========== TUTORIALS/GUIDES ==========
    {'prompt': 'First, we need to', 'desc': 'Step description - uncertain'},
    {'prompt': 'Next, you should', 'desc': 'Instruction - uncertain'},
    {'prompt': 'To get started,', 'desc': 'Getting started - uncertain'},
    {'prompt': 'In this tutorial, we will', 'desc': 'Tutorial goal - uncertain'},
    {'prompt': 'The recommended approach is to', 'desc': 'Recommendation - uncertain'},
    
    # ========== CREATIVE/METAPHORICAL ==========
    {'prompt': 'Write a poem about', 'desc': 'Creative topic - uncertain'},
    {'prompt': 'Write a poem about databases', 'desc': 'Poetry - uncertain'},
    {'prompt': 'If programming languages were animals, Python would be', 'desc': 'Metaphor - uncertain'},
    {'prompt': 'A story about a bug that', 'desc': 'Story - uncertain'},
    {'prompt': 'Imagine a world where code', 'desc': 'Imagination - uncertain'},
    {'prompt': 'Describe a good software engineer using nature metaphors', 'desc': 'Metaphor - uncertain'},
    
    # ========== GENERAL LANGUAGE ==========
    {'prompt': 'The weather today is', 'desc': 'Weather description - uncertain'},
    {'prompt': 'My favorite food is', 'desc': 'Preference - uncertain'},
    {'prompt': 'Once upon a time, there was', 'desc': 'Story beginning - uncertain'},
    {'prompt': 'In conclusion,', 'desc': 'Conclusion - uncertain'},
    {'prompt': 'On the other hand,', 'desc': 'Contrast - uncertain'},
    {'prompt': 'It is important to', 'desc': 'Importance - uncertain'},
    
    # ========== DEFINITIONS ==========
    {'prompt': 'Recursion is', 'desc': 'Definition - uncertain'},
    {'prompt': 'A closure is a function that', 'desc': 'Definition - uncertain'},
    {'prompt': 'Polymorphism allows', 'desc': 'Definition - uncertain'},
    {'prompt': 'Encapsulation refers to', 'desc': 'Definition - uncertain'},
    
    # ========== REASONING/ANALYSIS ==========
    {'prompt': 'This approach is preferred because', 'desc': 'Reasoning - uncertain'},
    {'prompt': 'The problem with this design is', 'desc': 'Problem - uncertain'},
    {'prompt': 'One potential issue is', 'desc': 'Issue - uncertain'},
    {'prompt': 'The trade-off here is', 'desc': 'Trade-off - uncertain'},
]

print(f"Language uncertainty examples: {len(LANGUAGE_UNCERTAINTY_EXAMPLES)}")

In [ ]:
# Cell 6: Combine and label all examples

TRAIN_EXAMPLES = []

# Add code uncertainty examples (label = 1)
for ex in CODE_UNCERTAINTY_EXAMPLES:
    TRAIN_EXAMPLES.append({
        'prompt': ex['prompt'],
        'label': 1,
        'desc': ex['desc'],
        'type': 'code_uncertainty'
    })

# Add language uncertainty examples (label = 0)
for ex in LANGUAGE_UNCERTAINTY_EXAMPLES:
    TRAIN_EXAMPLES.append({
        'prompt': ex['prompt'],
        'label': 0,
        'desc': ex['desc'],
        'type': 'language_uncertainty'
    })

print(f"\nTOTAL TRAINING EXAMPLES: {len(TRAIN_EXAMPLES)}")
print(f"  Code uncertainty (label=1): {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 1)}")
print(f"  Language uncertainty (label=0): {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 0)}")
print(f"\nClass balance: {sum(1 for e in TRAIN_EXAMPLES if e['label'] == 1) / len(TRAIN_EXAMPLES):.1%} code")

## 2. Extract Hidden States

In [ ]:
# Cell 7: Extract multi-layer hidden states

SELECTED_LAYERS = [8, 16, 31]  # Multi-layer for robust representation

def get_multi_layer_state(prompt: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
print(f"This will take a few minutes for {len(TRAIN_EXAMPLES)} examples...\n")

X_train = []
y_train = []

for example in tqdm(TRAIN_EXAMPLES, desc="Extracting"):
    h = get_multi_layer_state(example['prompt'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(example['label'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states extracted")
print(f"   Shape: {X_train.shape}")
print(f"   Labels: {y_train.shape}")
print(f"   Memory: {X_train.nbytes / 1024 / 1024:.1f} MB")

## 3. Train Classifier

In [ ]:
# Cell 8: Train with cross-validation

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Use 5-fold stratified CV (more robust than LOO for large dataset)
probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"✅ Multi-Layer Uncertainty Type Classifier")
print(f"   Layers: {SELECTED_LAYERS}")
print(f"   Hidden dim: {X_train.shape[1]:,}")
print(f"   Training examples: {len(y_train)}")
print(f"   5-Fold CV Accuracy: {cv_accuracy:.1%}")

# Detailed metrics
print(f"\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_train, 
    y_pred_cv,
    target_names=['Language Uncertainty', 'Code Uncertainty']
))

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_cv)
print(f"\nConfusion Matrix:")
print(f"                 Predicted LANG  Predicted CODE")
print(f"Actual LANG        {cm[0,0]:>6}          {cm[0,1]:>6}")
print(f"Actual CODE        {cm[1,0]:>6}          {cm[1,1]:>6}")

In [ ]:
# Cell 9: Analyze errors

print("\n" + "="*80)
print("ERROR ANALYSIS")
print("="*80)

errors = []
for i in range(len(y_train)):
    if y_train[i] != y_pred_cv[i]:
        errors.append({
            'prompt': TRAIN_EXAMPLES[i]['prompt'],
            'desc': TRAIN_EXAMPLES[i]['desc'],
            'true_label': y_train[i],
            'predicted': y_pred_cv[i]
        })

if errors:
    print(f"\nFound {len(errors)} errors ({len(errors)/len(y_train):.1%} of data):")
    print(f"\n{'Prompt':<40} {'Description':<30} {'True':<6} {'Pred'}")
    print("-"*80)
    for err in errors[:20]:  # Show first 20 errors
        true_str = 'CODE' if err['true_label'] == 1 else 'LANG'
        pred_str = 'CODE' if err['predicted'] == 1 else 'LANG'
        prompt_short = err['prompt'][:40]
        desc_short = err['desc'][:30]
        print(f"{prompt_short:<40} {desc_short:<30} {true_str:<6} {pred_str}")
    
    if len(errors) > 20:
        print(f"\n... and {len(errors) - 20} more errors")
else:
    print("\n✅ No errors! Perfect cross-validation accuracy.")

In [ ]:
# Cell 10: Train final model on all data

print("\nTraining final model on all data...")
probe.fit(X_train_scaled, y_train)

final_accuracy = probe.score(X_train_scaled, y_train)
print(f"✅ Final model trained")
print(f"   Training accuracy: {final_accuracy:.1%}")
print(f"   Coefficients shape: {probe.coef_.shape}")
print(f"   Intercept: {probe.intercept_[0]:.4f}")

## 4. Save Model

In [ ]:
# Cell 11: Save trained model

model_data = {
    'probe': probe,
    'scaler': scaler,
    'layers': SELECTED_LAYERS,
    'cv_accuracy': cv_accuracy,
    'train_accuracy': final_accuracy,
    'n_train_examples': len(y_train),
    'model_name': MODEL_NAME,
}

with open('uncertainty_type_classifier.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("✅ Model saved to: uncertainty_type_classifier.pkl")
print(f"\nModel summary:")
print(f"  - Layers: {SELECTED_LAYERS}")
print(f"  - Training examples: {len(y_train)}")
print(f"  - CV accuracy: {cv_accuracy:.1%}")
print(f"  - Final accuracy: {final_accuracy:.1%}")

# Save training data for reference
train_df = pd.DataFrame(TRAIN_EXAMPLES)
train_df.to_csv('training_data.csv', index=False)
print(f"\n✅ Training data saved to: training_data.csv")

## 5. Visualizations

In [ ]:
# Cell 12: Visualizations

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Confusion matrix heatmap
ax = axes[0, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred LANG', 'Pred CODE'],
            yticklabels=['True LANG', 'True CODE'])
ax.set_title(f'Confusion Matrix (Accuracy: {cv_accuracy:.1%})')

# Plot 2: Probability distribution
ax = axes[0, 1]
y_prob = cross_val_predict(probe, X_train_scaled, y_train, cv=cv, method='predict_proba')[:, 1]
code_probs = y_prob[y_train == 1]
lang_probs = y_prob[y_train == 0]
ax.hist(code_probs, bins=30, alpha=0.7, label='Code Uncertainty (true)', color='coral')
ax.hist(lang_probs, bins=30, alpha=0.7, label='Language Uncertainty (true)', color='steelblue')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision boundary')
ax.set_xlabel('P(Code Uncertainty)')
ax.set_ylabel('Count')
ax.set_title('Predicted Probability Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Top feature weights
ax = axes[1, 0]
weights = probe.coef_[0]
top_pos_idx = np.argsort(weights)[-10:][::-1]
top_neg_idx = np.argsort(weights)[:10]
top_weights = np.concatenate([weights[top_pos_idx], weights[top_neg_idx]])
colors_w = ['coral' if w > 0 else 'steelblue' for w in top_weights]
ax.barh(range(len(top_weights)), top_weights, color=colors_w, alpha=0.7)
ax.set_xlabel('Weight')
ax.set_title('Top 20 Feature Weights (Code+ / Language-)')
ax.axvline(0, color='black', linestyle='-', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')

# Plot 4: Error rate by prediction confidence
ax = axes[1, 1]
confidence = np.abs(y_prob - 0.5)  # Distance from decision boundary
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
bin_errors = []
bin_labels = []
for i in range(len(bins)-1):
    mask = (confidence >= bins[i]) & (confidence < bins[i+1])
    if mask.sum() > 0:
        error_rate = (y_train[mask] != y_pred_cv[mask]).mean()
        bin_errors.append(error_rate)
        bin_labels.append(f'{bins[i]:.1f}-{bins[i+1]:.1f}')
ax.bar(bin_labels, bin_errors, alpha=0.7, color='orange')
ax.set_xlabel('Confidence (distance from 0.5)')
ax.set_ylabel('Error Rate')
ax.set_title('Error Rate vs Prediction Confidence')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('uncertainty_classifier_training.png', dpi=150)
plt.show()

print("\n✅ Visualizations saved")

In [ ]:
# Cell 13: Summary

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)

print(f"\n✅ Uncertainty Type Classifier")
print(f"   Training examples: {len(y_train)}")
print(f"   Layers: {SELECTED_LAYERS}")
print(f"   Feature dimension: {X_train.shape[1]:,}")
print(f"   Cross-validation accuracy: {cv_accuracy:.1%}")
print(f"   Training accuracy: {final_accuracy:.1%}")

print(f"\n📁 Saved files:")
print(f"   - uncertainty_type_classifier.pkl (probe + scaler)")
print(f"   - training_data.csv (all training examples)")
print(f"   - uncertainty_classifier_training.png (visualizations)")

print(f"\n🚀 Next step:")
print(f"   Use this classifier in the interactive generation notebook!")
print(f"   The probe will monitor entropy during generation and classify")
print(f"   uncertainty type to decide when to stop and ask for clarification.")

print(f"\n" + "="*80)